# Training Artificial Neural Network (ANN)

## Preparing the Data

In [1]:
# for google colab to mount google drive
# from google.colab import drive
# drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [1]:
import pandas as pd
import numpy as np

In [2]:
# Loading Datasets
# '/content/gdrive/MyDrive/Colab Notebooks/raw/train/df_train.csv'
# '/content/gdrive/MyDrive/Colab Notebooks/raw/train/df_val.csv'

train_csv = '../data/raw/train/df_train.csv'
val_csv = '../data/raw/train/df_val.csv'
df_train = pd.read_csv(train_csv)
df_val = pd.read_csv(val_csv)

In [3]:
## Target Variable Processing

In [3]:
'''Target Variable Processing'''

target = 'price'
# Preparing the target Variable:
# The  y dataframes contain the Target Variable for training and validation
y_train = np.log1p(df_train[target])
y_val = np.log1p(df_val[target])

# Remove Target from training and validation datasets
X_train = df_train.drop(columns=target)
X_val = df_val.drop(columns=target)

In [4]:
# Defining column types
numerical = X_train.select_dtypes('number').columns.tolist()
categorical = X_train.select_dtypes(include='object').columns.tolist()
multi_label = [
    'comfort_convenience',
    'entertainment_media',
    'extras',
    'safety_security']

categorical = list(set(categorical)-set(multi_label))
categorical, numerical, multi_label

(['fuel',
  'drive_chain',
  'vat',
  'type',
  'gearing_type',
  'upholstery_type',
  'paint_type',
  'body_type',
  'make_model'],
 ['km',
  'gears',
  'age',
  'previous_owners',
  'hp_kw',
  'inspection_new',
  'displacement_cc',
  'weight_kg',
  'cons_comb'],
 ['comfort_convenience', 'entertainment_media', 'extras', 'safety_security'])

In [5]:
X_train = X_train[categorical + numerical + multi_label]
X_val = X_val[categorical + numerical + multi_label]

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction import DictVectorizer

In [7]:
class FlattenMultiLabelColumns(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        # Convert each row of the DataFrame into a dictionary;
        # then expand any list columns into multiple {key: 1} entries.
        rows = X.to_dict(orient='records')
        records = []
        for row in rows:
            new_row = {}
            for col, val in row.items():
                if col in multi_label and isinstance(val, list):
                    # For list columns, create new keys col=item
                    for item in val:
                        new_row[f"{col}={item}"] = 1
                else:
                    # For normal columns, just keep the original key/value
                    new_row[col] = val
            records.append(new_row)

        return records

In [8]:
# Apply Standard Scaler to numeric columns
transformer = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical)
    ],
    remainder='passthrough')

transformer.set_output(transform="pandas")

X_train = transformer.fit_transform(X_train)
X_val = transformer.transform(X_val)

In [9]:
X_val.head().T

,0,1,2,3,4
num__km,-0.221723,0.329085,0.524749,-0.287391,-0.192454
num__gears,0.076479,-1.350238,-1.350238,-1.350238,-1.350238
num__age,-0.378232,1.421108,0.521438,-0.378232,-0.378232
num__previous_owners,-0.1054,-0.1054,-0.1054,-0.1054,-0.1054
num__hp_kw,0.795067,-0.854934,-0.854934,-0.442433,-0.854934
num__inspection_new,1.700209,-0.588163,-0.588163,1.700209,1.700209
num__displacement_cc,-0.120044,0.103041,-0.037287,-1.562903,-0.037287
num__weight_kg,0.408834,-1.35575,-0.6878,-0.34884,-0.737647
num__cons_comb,0.888901,-2.108588,-1.647436,-0.494555,-1.070995
remainder__fuel,benzine,diesel,diesel,benzine,diesel


In [10]:
pipeline = Pipeline([
        ('flattern', FlattenMultiLabelColumns()),
        ('vectorizer', DictVectorizer(sparse=False))  # returns dense array
        ])

X_train =(pipeline.fit_transform(X_train))
X_val = pipeline.transform(X_val)

## Training the Neural Net

In [11]:
import tensorflow as tf
from tensorflow import keras

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [12]:
from keras.models import Sequential
from keras.layers import Dense, Dropout

In [14]:
model = Sequential()
model.add(Dense(30, activation='relu', input_shape=(X_train.shape[1], )))
model.add(Dropout(0.1))
model.add(Dense(1))

/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [15]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 30)                  │         246,810 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 30)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │              31 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 246,841 (964.22 KB)

 Trainable params: 246,841 (964.22 KB)

 Non-trainable params: 0 (0.00 B)

In [16]:
learning_rate = 0.01
optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
loss = keras.losses.MeanSquaredError()
rmse = keras.metrics.RootMeanSquaredError()

In [ ]:
model.compile(optimizer=optimizer, loss=loss, metrics=[rmse])

In [17]:
history = model.fit(
    X_train,
    y_train,
    batch_size=1024,
    epochs=10,
    verbose=1,
    validation_data=(X_val,y_val)
    )

In [ ]:
import matplotlib.pyplot as plt

def plot_metrics(history):
    # Plot training & validation loss values
    plt.figure(figsize=(10, 5))
    plt.plot(history.history['loss'])
    plt.plot(history.history['val_loss'])
    plt.title('Model loss')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train', 'Validation'], loc='upper left')
    plt.show()

    # Plot training & validation RMSE values
    plt.figure(figsize=(10, 5))
    plt.plot(history.history['root_mean_squared_error'])
    plt.plot(history.history['val_root_mean_squared_error'])
    plt.title('Model RMSE')
    plt.ylabel('RMSE')
    plt.xlabel('Epoch')
    plt.legend(['Train', 'Validation'], loc='upper left')
    plt.show()

    # Print the history
    history.history


In [ ]:
plot_metrics(history)

## Optimization

### Adjusting learning Rate

In [ ]:
scores = {}

for learning_rate in [0.001, 0.01, 0.1]:
    print(learning_rate)
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss=loss, metrics=[rmse])
    history = history = model.fit(
        X_train,
        y_train,
        batch_size=1024,
        epochs=10,
        verbose=1,
        validation_data=(X_val,y_val)
    )
    scores[learning_rate] = history.history
    print('\n\n')


In [ ]:

for learning_rate, history in scores.items():
    print(learning_rate)
    plot_metrics(history)

In [ ]:
# Clear the session to reset any internal states in TensorFlow/Keras.
keras.backend.clear_session()

### Adding more layers

In [13]:
model = Sequential()
model.add(Dense(64, activation='relu',input_shape=(X_train.shape[1], )))
model.add(Dropout(0.15))
model.add(Dense(128, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(16, activation='relu'))
model.add(Dense(1))

/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [14]:
learning_rate = 0.01
optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
loss = keras.losses.MeanSquaredError()
rmse = keras.metrics.RootMeanSquaredError()

In [ ]:
model.compile(optimizer=optimizer, loss=loss, metrics=[rmse])
model.save_weights('model_v1.h5', save_format='h5')

In [ ]:
# Adding Checkpoints at which to save the model.
checkpoint = keras.callbacks.ModelCheckpoint(
        'ann_v1_{epoch:02d}_{val_root_mean_squared_error:.3f}.h5',
        save_best_only=True,
        monitor='val_accuracy',
        mode='max'
    )

In [15]:
history = model.fit(
        X_train,
        y_train,
        batch_size=4096,
        epochs=50,
        verbose=1,
        validation_data=(X_val,y_val),
        callback=[checkpoint]
)

Epoch 1/45
9/9 ━━━━━━━━━━━━━━━━━━━━ 6s 378ms/step - loss: 62.8311 - root_mean_squared_error: 7.8328 - val_loss: 9.9507 - val_root_mean_squared_error: 3.1545
Epoch 2/45
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 9.1574 - root_mean_squared_error: 3.0190 - val_loss: 6.8257 - val_root_mean_squared_error: 2.6126
Epoch 3/45
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - loss: 7.1370 - root_mean_squared_error: 2.6551 - val_loss: 2.4817 - val_root_mean_squared_error: 1.5753
Epoch 4/45
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 46ms/step - loss: 2.5898 - root_mean_squared_error: 1.6005 - val_loss: 1.1912 - val_root_mean_squared_error: 1.0914
Epoch 5/45
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 45ms/step - loss: 0.9600 - root_mean_squared_error: 0.9778 - val_loss: 0.3815 - val_root_mean_squared_error: 0.6176
Epoch 6/45
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - loss: 1.0494 - root_mean_squared_error: 1.0237 - val_loss: 0.2653 - val_root_mean_squared_error: 0.5151
Epoch 7/45
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - loss: 0.7568 - ro

In [ ]:
plot_metrics(history)